<a href="https://colab.research.google.com/github/yanaguntikarmeesal/Cat-vs-Dog-image-classification/blob/main/M_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cat vs Dog image classification using VGG16 transfer learning.

# 1. Install / import libraries

In [ ]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Sequential

# 2. Check TensorFlow

In [ ]:
# ============================================================
# CELL 2: CHECK TENSORFLOW
# ============================================================

print("TensorFlow Version:", tf.__version__)

# 3. Check GPU

In [ ]:
# ============================================================
# CELL 3: CHECK GPU
# ============================================================

print("GPU Available:", tf.config.list_physical_devices('GPU'))

# 4. Upload your dataset

# 5. Upload ZIP

In [ ]:
# ============================================================
# CELL 4: UPLOAD DATASET ZIP
# ============================================================

from google.colab import files

uploaded = files.upload()

# 6. Extract ZIP

In [ ]:
# ============================================================
# CELL 5: EXTRACT DATASET
# ============================================================

import zipfile
import os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("Dataset extracted successfully!")

# 7. Find dataset folders

In [ ]:
# ============================================================
# CELL 6: CHECK DATASET
# ============================================================

for root, dirs, files_list in os.walk('/content'):
    if 'train' in dirs and 'validation' in dirs:
        print("Dataset found at:", root)

In [ ]:
# ============================================================
# CELL 7: DATASET PATHS
# ============================================================

train_data_dir = "/content/Data/train"
validation_data_dir = "/content/Data/validation"

print("Training directory:", train_data_dir)
print("Validation directory:", validation_data_dir)

# 8. Check classes

In [ ]:
# ============================================================
# CELL 8: CHECK CLASSES
# ============================================================

print("Training classes:")
print(os.listdir(train_data_dir))

print("\nValidation classes:")
print(os.listdir(validation_data_dir))

# 9. Count images

In [ ]:
# ============================================================
# CELL 9: COUNT IMAGES
# ============================================================

def count_images(folder):

    total = 0

    for class_name in os.listdir(folder):

        class_path = os.path.join(folder, class_name)

        if os.path.isdir(class_path):

            count = len([
                f for f in os.listdir(class_path)
                if f.lower().endswith(
                    ('.jpg', '.jpeg', '.png', '.bmp', '.gif')
                )
            ])

            print(class_name, ":", count)

            total += count

    print("Total images:", total)


print("TRAINING DATA")
count_images(train_data_dir)

print("\nVALIDATION DATA")
count_images(validation_data_dir)

# 10. Load training dataset

This follows the same approach as your PDF, using image_dataset_from_directory, image size (224,224), inferred labels, integer labels and batch size 32

In [ ]:
# ============================================================
# CELL 10: LOAD TRAINING DATA
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_data_generator = tf.keras.utils.image_dataset_from_directory(
    train_data_dir,
    image_size=IMG_SIZE,
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Training dataset loaded successfully.")

# 11. Load validation dataset

In [ ]:
# ============================================================
# CELL 11: LOAD VALIDATION DATA
# ============================================================

validation_data_generator = tf.keras.utils.image_dataset_from_directory(
    validation_data_dir,
    image_size=IMG_SIZE,
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Validation dataset loaded successfully.")

# 12. Check class names

In [ ]:
# ============================================================
# CELL 12: CLASS NAMES
# ============================================================

class_names = train_data_generator.class_names

print("Classes:", class_names)

# 13. Display sample images

In [ ]:
# ============================================================
# CELL 13: DISPLAY TRAINING IMAGES
# ============================================================

plt.figure(figsize=(12, 8))

for images, labels in train_data_generator.take(1):

    # Iterate up to the number of images in the batch, or a maximum of 9
    for i in range(min(9, len(images))):

        ax = plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        plt.title(class_names[labels[i].numpy()])

        plt.axis("off")

plt.tight_layout()
plt.show()

# 14. Improve dataset performance

In [ ]:
# ============================================================
# CELL 14: PREFETCH DATA
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_data_generator = train_data_generator.prefetch(
    buffer_size=AUTOTUNE
)

validation_data_generator = validation_data_generator.prefetch(
    buffer_size=AUTOTUNE
)

# 15. Load VGG16

Your original project uses VGG16 with:

In [ ]:
# ============================================================
# CELL 15: LOAD VGG16
# ============================================================

base_model = VGG16(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)

print("VGG16 loaded successfully.")

# 16. Freeze VGG16 layers

In [ ]:
# ============================================================
# CELL 16: FREEZE VGG16
# ============================================================

for layer in base_model.layers:
    layer.trainable = False

print("All VGG16 layers frozen.")

In [ ]:
# ============================================================
# CELL 17: CHECK TRAINABLE LAYERS
# ============================================================

for layer in base_model.layers:

    print(
        layer.name,
        "Trainable:",
        layer.trainable
    )

# 17. VGG16 summary

In [ ]:
# ============================================================
# CELL 18: VGG16 SUMMARY
# ============================================================

base_model.summary()

# 18. Build Cat vs Dog model


1. VGG16
↓
2. Flatten
↓
3. Dense(256, ReLU)
↓
4. Dense(1, Sigmoid)

In [ ]:
# ============================================================
# CELL 19: CREATE FINAL MODEL
# ============================================================

model = Sequential()

# VGG16 base model
model.add(base_model)

# Flatten feature maps
model.add(Flatten())

# Fully connected layer
model.add(
    Dense(
        256,
        activation='relu'
    )
)

# Output layer
model.add(
    Dense(
        1,
        activation='sigmoid'
    )
)

model.summary()

# 19. Compile model

The original PDF uses binary cross-entropy, Adam and accuracy

In [ ]:
# ============================================================
# CELL 20: COMPILE MODEL
# ============================================================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")

# 20. Train model

Your original project trains for 5 epochs

In [ ]:
# ============================================================
# CELL 21: TRAIN MODEL
# ============================================================

history = model.fit(
    train_data_generator,
    validation_data=validation_data_generator,
    epochs=5,
    verbose=1
)

# 21. Plot training accuracy

In [ ]:
# ============================================================
# CELL 22: TRAINING ACCURACY GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title('Training vs Validation Accuracy')

plt.legend()

plt.grid()

plt.show()

# 22. Plot loss

In [ ]:
# ============================================================
# CELL 23: TRAINING LOSS GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title('Training vs Validation Loss')

plt.legend()

plt.grid()

plt.show()

# 23. Evaluate model

In [ ]:
# ============================================================
# CELL 24: MODEL EVALUATION
# ============================================================

loss, accuracy = model.evaluate(
    validation_data_generator
)

print("Validation Loss:", loss)

print("Validation Accuracy:", accuracy)

# 24. Upload a test image

In [ ]:
# ============================================================
# CELL 25: UPLOAD TEST IMAGE
# ============================================================

from google.colab import files

uploaded_test = files.upload()

test_image_path = list(uploaded_test.keys())[0]

print("Selected image:", test_image_path)

# 25. Display test image

In [ ]:
# ============================================================
# CELL 26: DISPLAY TEST IMAGE
# ============================================================

import cv2

img = cv2.imread(test_image_path)

img_rgb = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(6, 6))

plt.imshow(img_rgb)

plt.title("Test Image")

plt.axis("off")

plt.show()

# 26. Check image shape

In [ ]:
# ============================================================
# CELL 27: IMAGE SHAPE
# ============================================================

print("Original image shape:", img.shape)

# 27. Create prediction function

Your PDF uses a function that resizes an image to (224,224), reshapes it, and calls model.predict().

In [ ]:
# ============================================================
# CELL 28: PREDICTION FUNCTION
# ============================================================

def predict_image(image_path, model):

    # Read image
    img = cv2.imread(image_path)

    # Check image
    if img is None:
        print("Unable to read image.")
        return None

    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    # Resize
    img_resized = cv2.resize(
        img_rgb,
        (224, 224)
    )

    # Convert to float
    img_array = np.array(
        img_resized,
        dtype=np.float32
    )

    # Add batch dimension
    img_array = np.expand_dims(
        img_array,
        axis=0
    )

    # Prediction
    prediction = model.predict(
        img_array,
        verbose=0
    )[0][0]

    # Classification
    if prediction >= 0.5:
        result = "Dog"
        confidence = prediction
    else:
        result = "Cat"
        confidence = 1 - prediction

    return result, confidence

# 28. Predict uploaded image

In [ ]:
# ============================================================
# CELL 29: PREDICT IMAGE
# ============================================================

result, confidence = predict_image(
    test_image_path,
    model
)

print("Prediction:", result)

print(
    "Confidence:",
    f"{confidence * 100:.2f}%"
)

# 29. Display prediction

In [ ]:
# ============================================================
# CELL 30: DISPLAY PREDICTION
# ============================================================

plt.figure(figsize=(7, 7))

plt.imshow(img_rgb)

plt.title(
    f"{result} - Confidence: {confidence * 100:.2f}%"
)

plt.axis("off")

plt.show()

# 30. Save trained model

Your original PDF saves the model as catvsdog.h5

In [ ]:
# ============================================================
# CELL 31: SAVE MODEL
# ============================================================

model.save(
    "/content/catvsdog.h5"
)

print("Model saved successfully!")

# 31. Download model to your computer

In [ ]:
# ============================================================
# CELL 32: DOWNLOAD MODEL
# ============================================================

from google.colab import files

files.download(
    "/content/catvsdog.h5"
)

# 32. Complete prediction test

You can upload another image and test it:

In [ ]:
# ============================================================
# CELL 33: TEST ANOTHER IMAGE
# ============================================================

uploaded_test = files.upload()

test_image_path = list(uploaded_test.keys())[0]

result, confidence = predict_image(
    test_image_path,
    model
)

print("================================")
print("       CAT vs DOG RESULT")
print("================================")
print("Image:", test_image_path)
print("Prediction:", result)
print(
    "Confidence:",
    f"{confidence * 100:.2f}%"
)
print("================================")